# 15 - Qwen Text + Video (SigLIP)
Modelo multimodal video + transcripcion con Qwen2.5 para texto y SigLIP para video.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, cross_val_score
from transformers import AutoTokenizer, AutoModel, AutoProcessor
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
SEED = 42
GROUP_ID = 'BeingChillingWeWillWin'
MODEL_TAG = 'QwenText_SigLIPVideo_XGB_ES_EN'
TEST_CASE = 'EXIST2026'
TRAIN_LANGS = ['es', 'en']
TEXT_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_TEXT_LEN = 256
TEXT_BATCH = 8
PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_JSON = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json'
CLUSTER_JSON = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json')
TRAIN_JSON_PATH = LOCAL_JSON if LOCAL_JSON.exists() else CLUSTER_JSON
CLUSTER_ROOT = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset')
VIDEO_ROOTS = [TRAIN_JSON_PATH.parent, TRAIN_JSON_PATH.parent.parent, CLUSTER_ROOT / 'training', CLUSTER_ROOT]

def resolve_video_abs_path(path_video: str) -> str:
    pv = Path(str(path_video))
    if pv.is_absolute() and pv.exists():
        return str(pv.resolve())
    cands = []
    for root in VIDEO_ROOTS:
        if root.exists():
            cands.extend([(root / pv).resolve(), (root / 'training' / pv).resolve()])
    for c in cands:
        if c.exists():
            return str(c)
    return str(cands[0] if cands else pv.resolve())

def majority_task3_1(label_list):
    vals = [x for x in label_list if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    return vals[0] if c['YES'] == c['NO'] else c.most_common(1)[0][0]

with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)
df = pd.DataFrame([{'sample_id': str(k), **v} for k, v in raw.items()])
df['text'] = df['text'].fillna('').astype(str)
df['y'] = df['labels_task3_1'].apply(majority_task3_1).map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df['video_abs_path'] = df['path_video'].apply(resolve_video_abs_path)
train_mask = np.isin(df['lang'].astype(str).to_numpy(), TRAIN_LANGS) & (df['y'].to_numpy() >= 0)
split_upper = df['split'].astype(str).str.upper() if 'split' in df.columns else pd.Series([''] * len(df))
test_mask = split_upper.str.contains('TEST').to_numpy()
infer_mask = test_mask if test_mask.any() else ((df['y'].to_numpy() < 0) if (df['y'].to_numpy() < 0).any() else (~train_mask if (~train_mask).any() else np.ones(len(df), dtype=bool)))
work = df.loc[train_mask | infer_mask, ['sample_id', 'video_abs_path', 'text', 'y']].reset_index(drop=True)
print('Rows for run:', len(work))

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
video_processor = AutoProcessor.from_pretrained('google/siglip-base-patch16-224')
video_model = AutoModel.from_pretrained('google/siglip-base-patch16-224').to(DEVICE)
video_model.eval()

def sample_frames(video_path, n=8):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return []
    idxs = np.linspace(0, max(total - 1, 0), num=n, dtype=int)
    out = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok and frame is not None:
            out.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return out

def embed_video(path):
    if not Path(path).exists():
        return None
    frames = sample_frames(path, 8)
    if not frames:
        return None
    inp = video_processor(images=frames, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    with torch.no_grad():
        feat = video_model.get_image_features(**inp)
    return feat.mean(dim=0).detach().cpu().numpy().astype(np.float32)

work['video_emb'] = [embed_video(p) for p in tqdm(work['video_abs_path'], desc='Video embeddings')]
work = work.loc[work['video_emb'].notnull()].reset_index(drop=True)

tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID, trust_remote_code=True)
dtype = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32
text_model = AutoModel.from_pretrained(TEXT_MODEL_ID, torch_dtype=dtype, trust_remote_code=True, device_map='auto')
text_model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

text_chunks = []
texts = work['text'].tolist()
for start in tqdm(range(0, len(texts), TEXT_BATCH), desc='Qwen text embeddings'):
    batch = texts[start:start + TEXT_BATCH]
    enc = tok(batch, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    enc = {k: v.to(text_model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = text_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc['attention_mask'])
    text_chunks.append(pooled.detach().cpu().numpy().astype(np.float32))

X_text = np.vstack(text_chunks)
X_video = np.vstack(work['video_emb'].to_list()).astype(np.float32)
X = np.hstack([X_text, X_video]).astype(np.float32)
y = work['y'].to_numpy(dtype=np.int64)
ids = work['sample_id'].astype(str).to_numpy()
train_ids = set(df.loc[train_mask, 'sample_id'].astype(str).tolist())
infer_ids_set = set(df.loc[infer_mask, 'sample_id'].astype(str).tolist())
is_train = np.array([x in train_ids for x in ids], dtype=bool)
is_infer = np.array([x in infer_ids_set for x in ids], dtype=bool)
X_train, y_train = X[is_train], y[is_train]
X_infer, infer_ids = X[is_infer], ids[is_infer]
clf = XGBClassifier(n_estimators=700, max_depth=6, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8, objective='binary:logistic', eval_metric='logloss', random_state=SEED, n_jobs=-1) if HAS_XGB else HistGradientBoostingClassifier(learning_rate=0.05, max_depth=10, random_state=SEED)
scores = cross_val_score(clf, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), scoring='f1_macro', n_jobs=-1)
print('CV F1 macro mean:', float(scores.mean()))
clf.fit(X_train, y_train)
pred_labels = np.where(clf.predict(X_infer).astype(int) == 1, 'YES', 'NO')

In [ ]:
import joblib
model_path = WEIGHTS_DIR / f'{GROUP_ID}_{MODEL_TAG}.joblib'
joblib.dump(clf, model_path)
rows = [{'test_case': TEST_CASE, 'id': str(sid), 'value': str(lab)} for sid, lab in zip(infer_ids, pred_labels)]
json_path = DELIVERABLES_DIR / f'{GROUP_ID}_{MODEL_TAG}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False)
print('Saved model:', model_path)
print('Saved json:', json_path)
print('Rows:', len(rows))